## Data Ingestion and Chunking (page level)

In [ ]:
from pypdf import PdfReader
from docx import Document
from docx2pdf import convert

def process_file(file_path: str):
    chunks = []
    
        
    # 1.processing pdf page by page
    if file_path.endswith(('.pdf' or '.docx')):
        if file_path.endswith('.docx'):
            pdf_to_read = file_path.replace('.docx', '_temp.pdf')
            convert(file_path, pdf_to_read)
            # temp_pdf_created = True
        reader = PdfReader(file_path)
        for i, page in enumerate(reader.pages):
            text = (page.extract_text() or "").strip()
            if text:
                # page level chunking with a 5000 char cap
                chunk_text = text[:5000]
                chunks.append({
                    "text": chunk_text,
                    "source": f"Page {i + 1}"
                })

            if len(text) > 5000:
                chunk_text = text[5000:]
                chunks.append({
                    "text": chunk_text,
                    "source": f"Page {i + 2}"
                })
                
    # # 2.for processing docx
    # elif file_path.endswith('.docx'):
    #     doc = Document(file_path)
    #     full_text = "\n".join([p.text for p in doc.paragraphs if p.text.strip()])
    #     for i in range(0, len(full_text), 5000):
    #         chunks.append({
    #             "text": full_text[i:i+5000],
    #             "source": f"Section {i//5000 + 1}"
    #         })
    #will simply convert the pdfs to the docx no need to have separate logic
            
    return chunks
 

In [2]:
import os
import pandas as pd
from pypdf import PdfReader
from pptx import Presentation
from docx2pdf import convert

def process_file(file_path: str):
    chunks = []

    if file_path.endswith('.xlsx'):
        xl = pd.ExcelFile(file_path)
        for sheet_name in xl.sheet_names:
            # df = xl.parse(sheet_name, header=3)
            df = xl.parse(sheet_name)
            for i, row in df.iterrows():
                row_str = " | ".join([f"{col}: {val}" for col, val in row.items() if pd.notna(val)])
                if row_str.strip():
                    for start in range(0, len(row_str), 5000):
                        part_num = start // 5000 + 1
                        chunks.append({
                            "text": row_str[start:start+5000],
                            "source": f"Sheet '{sheet_name}', Row {i + 2} (part {part_num})"
                        })

    # 2. PPTX Handler
    elif file_path.endswith('.pptx'):
        prs = Presentation(file_path)
        for i, slide in enumerate(prs.slides):
            text = []
            # for the all slide components like text frames , tables containg the text etc, 
            for shape in slide.shapes:
                if shape.has_text_frame:
                    for p in shape.text_frame.paragraphs:
                        if p.text.strip():
                            text.append(p.text.strip())
                elif shape.has_table:
                    table = shape.table
                    for row in table.rows:
                        row_text = " | ".join(cell.text.strip() for cell in row.cells if cell.text.strip())
                        if row_text:
                            text.append(row_text)
            #incase there're any speaker notes
            if slide.has_notes_slide:
                notes_text = slide.notes_slide.notes_text_frame.text.strip()
                if notes_text:
                    text.append("Notes: " + notes_text)
            full_text = "\n".join(text)
            if full_text:
                for start in range(0, len(full_text), 5000):
                    part_num = start // 5000 + 1
                    chunks.append({"text": full_text[start:start+5000], "source": f"Slide {i + 1} (part {part_num})"})

    # 3. PDF & DOCX Handler
    elif file_path.endswith(('.pdf', '.docx')):
        pdf_to_read = file_path
        temp_pdf = False
        if file_path.endswith('.docx'):
            pdf_to_read = file_path.replace('.docx', '_temp.pdf')
            convert(file_path, pdf_to_read)
            temp_pdf = True

        reader = PdfReader(pdf_to_read)
        for i, page in enumerate(reader.pages):
            text = (page.extract_text() or "").strip()
            if text:
                for start in range(0, len(text), 5000):
                    part_num = start // 5000 + 1
                    chunks.append({"text": text[start:start+5000], "source": f"Page {i + 1} (part {part_num})"})

        if temp_pdf and os.path.exists(pdf_to_read):
            os.remove(pdf_to_read)

    return chunks

In [7]:
import pandas as pd
# xl = pd.ExcelFile("DSA_PracticeQs.xlsx")
xl = pd.ExcelFile("test_multi_sheet.xlsx")
print(xl.sheet_names)

['Employee Directory', 'Project Milestones', 'Q3 Budget Breakdown']


In [3]:
# chunks = process_file("sample.pdf")
# chunks = process_file("changeControl.pptx")
chunks = process_file("../data/uploads/changeControl.pptx")
# chunks = process_file("CarromRules.docx")
# chunks = process_file("DSA_PracticeQs.xlsx")
#chunks = process_file("test_multi_sheet.xlsx")
print(f"Total chunks extracted: {len(chunks)}")
print("\nFirst Chunk: ")
print(f"Source: {chunks[7]['source']}")
print(f"Content:\n{chunks[7]['text'][:600]}...")

Total chunks extracted: 13

First Chunk: 
Source: Slide 8 (part 1)
Content:
80%
REDUCED RISK
Strategic Benefits
Strong governance leads to better risk management and resource allocation.
•
Accountability: Transparent decision making.
•
Efficiency: Less waste on irrelevant IT projects.
•
Value: Maximizing return on technology investments.
Why Invest in Governance?...


## Vector Embedding and using Milvus Storage

In [27]:
from pymilvus import MilvusClient
from sentence_transformers import SentenceTransformer

In [28]:
# 1.loading an embedding model
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
VECTOR_DIMENSION = 384 

# 2.connecting to milvus local database
client = MilvusClient("rag_milvus.db")
COLLECTION_NAME = "page_chunks_collection"

#resets collection for clean testing
if client.has_collection(collection_name=COLLECTION_NAME):
    client.drop_collection(collection_name=COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=VECTOR_DIMENSION,
    metric_type="COSINE" 
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [29]:
# 3.generating Embeddings and Format Payload
texts = [item["text"] for item in chunks]
embeddings = embedding_model.encode(texts)

data_to_insert = [
    {
        "id": idx,
        "vector": embeddings[idx].tolist(),
        "text": chunks[idx]["text"],
        "source": chunks[idx]["source"]
    }
    for idx in range(len(chunks))
]

# 4.inserting into milvus
insert_result = client.insert(collection_name=COLLECTION_NAME, data=data_to_insert)
print(f"Ingestion complete. Inserted count: {insert_result['insert_count']}")

Ingestion complete. Inserted count: 10


## Query Embedding

In [42]:
# test_query = "What are atomic habits?"
# test_query = "How much budget remains for Software & Cloud?"
test_query = "Who is leading the Enterprise RAG Pipeline project, and what is its budget?"

In [43]:
query_vector = embedding_model.encode([test_query])

## Vector Search

In [44]:
#finding the top 5 most semantically relevant chunks
search_results = client.search(
    collection_name=COLLECTION_NAME,
    data=query_vector.tolist(),
    limit=5,  
    output_fields=["text", "source"]
)

## Retrieving chunks

In [45]:
retrieved_chunks = []

for ch in search_results[0]:
    source = ch['entity']['source']
    text = ch['entity']['text']
    #cosin simialarity score
    score = ch['distance']
    
    info = "[" + source + "] (similarity: " + str(score) + "):\n" + text
    retrieved_chunks.append(info)

In [46]:
#combining them into a single one str
context_block = "\n-----\n".join(retrieved_chunks)

print("Reterived chunk contxt\n")
print(context_block)

Reterived chunk contxt

[Sheet 'Project Milestones', Row 2 (part 1)] (similarity: 0.6874191761016846):
Project ID: PRJ-01 | Project Name: Enterprise RAG Pipeline | Lead: Bilal Ahmed | Budget ($): 15000 | Deadline: 2026-09-30 | Status: In Progress
-----
[Sheet 'Project Milestones', Row 3 (part 1)] (similarity: 0.39732110500335693):
Project ID: PRJ-02 | Project Name: HR Portal Migration | Lead: Chaudhry Riaz | Budget ($): 8000 | Deadline: 2026-10-15 | Status: Planning
-----
[Sheet 'Project Milestones', Row 4 (part 1)] (similarity: 0.3958166241645813):
Project ID: PRJ-03 | Project Name: Marketing Dashboard | Lead: Dania Zafar | Budget ($): 5000 | Deadline: 2026-08-31 | Status: Completed
-----
[Sheet 'Q3 Budget Breakdown', Row 4 (part 1)] (similarity: 0.3594512343406677):
Category: Training & Workshops | Description: AI Team Onboarding | Allocated ($): 3000 | Spent ($): 1200 | Remaining ($): 1800
-----
[Sheet 'Q3 Budget Breakdown', Row 3 (part 1)] (similarity: 0.333373486995697):
Category:

## LLM Generation
#### (Groq api integrated)

In [47]:
import os
from dotenv import load_dotenv
from groq import Groq

In [48]:
load_dotenv()
groq_client = Groq()

In [49]:
#defining system instructions n the user prompt
system_instruction = """You are an accurate, factual AI document assistant.
Answer the user's question using ONLY the provided document context.
If the information is not explicitly in the context, state: 'I cannot answer based on the provided document.'"""

user_prompt = f"""--- RETRIEVED CONTEXT ---
{context_block}

--- USER QUESTION ---
{test_query}"""

In [50]:
#Calling Groq API
chat_completion = groq_client.chat.completions.create(
    messages=[
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_prompt}
    ],
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

In [51]:
llm_answer = chat_completion.choices[0].message.content

print("=" * 60)
print("LLM GENERATED ANSWER")
print("=" * 60)
print(llm_answer)

print("\n" + "=" * 60)
print("VERIFICATION: RETRIEVED CHUNKS USED")
print("=" * 60)
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"\n--- Chunk#{i} ---")
    print(chunk)

LLM GENERATED ANSWER
The lead for the Enterprise RAG Pipeline project is Bilal Ahmed, and its budget is $15,000.

VERIFICATION: RETRIEVED CHUNKS USED

--- Chunk#1 ---
[Sheet 'Project Milestones', Row 2 (part 1)] (similarity: 0.6874191761016846):
Project ID: PRJ-01 | Project Name: Enterprise RAG Pipeline | Lead: Bilal Ahmed | Budget ($): 15000 | Deadline: 2026-09-30 | Status: In Progress

--- Chunk#2 ---
[Sheet 'Project Milestones', Row 3 (part 1)] (similarity: 0.39732110500335693):
Project ID: PRJ-02 | Project Name: HR Portal Migration | Lead: Chaudhry Riaz | Budget ($): 8000 | Deadline: 2026-10-15 | Status: Planning

--- Chunk#3 ---
[Sheet 'Project Milestones', Row 4 (part 1)] (similarity: 0.3958166241645813):
Project ID: PRJ-03 | Project Name: Marketing Dashboard | Lead: Dania Zafar | Budget ($): 5000 | Deadline: 2026-08-31 | Status: Completed

--- Chunk#4 ---
[Sheet 'Q3 Budget Breakdown', Row 4 (part 1)] (similarity: 0.3594512343406677):
Category: Training & Workshops | Description: